In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [2]:
np.random.seed(42)

In [3]:
n_members=  50_000

In [4]:
members_id = np.array([f"M{i}" for i in range (n_members)])

In [5]:
age_groups = [
    (0, 18, 0.18),
    (19,34,0.28),
    (35,49,0.26),
    (50, 64, 0.22),
    (65,75,0.06)
]

In [6]:
ages =[]

In [7]:
for low, high, prob in age_groups:
  count = int (prob* n_members)
  ages.append (np.random.randint (low,high+1, count))
ages= np.concatenate(ages)
np.random.shuffle(ages)


In [8]:
if len(ages)>n_members:
  ages = ages[:n_members]

In [9]:
genders = np.random.choice (["M", "F"], size=n_members)

In [10]:
regions = np.random.choice (["North", "South", "East", "West", "Central", "Costal"], size = n_members)

In [11]:
members =  pd.DataFrame ({"member_id": members_id,"age": ages, "gender": genders, "region":regions  })

In [12]:
members.sample(10)

,member_id,age,gender,region
29033,M29033,69,F,East
15995,M15995,16,F,Costal
34452,M34452,61,M,North
35943,M35943,1,F,East
29011,M29011,10,F,North
1130,M1130,43,F,Costal
43846,M43846,5,F,Costal
34971,M34971,44,M,North
38410,M38410,29,M,North
19917,M19917,27,M,Costal


In [13]:
def chronic_prob (age):
  if age<30:
    return 0.10
  elif age <50:
    return 0.25
  elif age <65:
    return 0.45
  else:
    return 0.65



In [14]:
probs = np.array([chronic_prob(a) for a in members ["age"]])

In [15]:
members["chronic_count"] = np.random.poisson (lam = probs *2)

In [16]:
members.sample(10)

,member_id,age,gender,region,chronic_count
12990,M12990,54,F,Central,0
15569,M15569,48,M,West,0
3472,M3472,15,F,West,2
9603,M9603,60,F,Central,0
29223,M29223,49,M,Central,1
43423,M43423,12,F,South,0
2050,M2050,21,F,West,1
43082,M43082,41,F,East,2
39348,M39348,58,F,North,2
959,M959,33,F,Costal,1


In [17]:
members["behavioral_flag"] =  np.random.binomial (1,0.15,n_members)

In [18]:
members.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag
39035,M39035,45,M,North,0,0
38369,M38369,23,F,East,0,1
8366,M8366,74,F,South,1,0
38547,M38547,38,M,South,1,0
2687,M2687,4,M,West,0,0
6389,M6389,52,M,Central,1,0
15032,M15032,25,M,West,0,0
5506,M5506,16,M,Central,0,1
40679,M40679,42,F,South,0,0
26648,M26648,28,M,Costal,0,1


In [19]:
members = members.drop(columns=["risk_level"], errors="ignore")

In [20]:
members["risk_score"] = (
    members["chronic_count"] * 2 +
    members["behavioral_flag"] * 1 +
    np.random.normal(0, 1, len(members))
)


In [21]:
members["risk_level"] = pd.qcut(
    members["risk_score"],
    q=[0, 0.65, 0.90, 1.0],
    labels=["Low", "Medium", "High"]
)


In [22]:
print(members["risk_level"].value_counts(normalize=True))

risk_level
Low       0.65
Medium    0.25
High      0.10
Name: proportion, dtype: float64


In [23]:
conditions = [members ["chronic_count"]>=4,members ["chronic_count"]>2]

In [24]:
choices = ["High", "Medium"]

In [25]:
members["risk_level"].value_counts(normalize=True)

,proportion
risk_level,
Low,0.65
Medium,0.25
High,0.10


In [26]:
members.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level
27408,M27408,27,M,South,0,0,-0.233885,Low
23515,M23515,45,M,West,0,0,0.138000,Low
41001,M41001,43,M,North,0,0,1.441608,Low
31282,M31282,30,F,Costal,0,0,-1.226103,Low
34829,M34829,41,M,Central,1,1,2.877879,Medium
19637,M19637,53,F,Central,0,0,0.679744,Low
4932,M4932,61,M,West,1,0,2.210257,Medium
21004,M21004,12,F,West,0,0,-1.035986,Low
18371,M18371,31,F,South,0,0,-0.369127,Low
854,M854,4,M,South,0,0,-1.894137,Low


In [27]:
members ["risk_level"].value_counts(normalize= True)

,proportion
risk_level,
Low,0.65
Medium,0.25
High,0.10


In [28]:
members.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level
13338,M13338,38,M,Central,0,0,0.001843,Low
30733,M30733,14,M,West,0,0,-0.379874,Low
26313,M26313,19,F,North,1,1,2.677272,Medium
12579,M12579,32,F,North,2,0,3.728148,Medium
49285,M49285,12,M,South,0,0,-1.571118,Low
44915,M44915,20,M,North,0,0,0.518994,Low
46931,M46931,31,F,North,0,0,0.437179,Low
32016,M32016,19,F,Costal,0,0,1.262675,Low
10882,M10882,25,M,North,0,0,-0.308771,Low
36629,M36629,43,F,South,0,0,0.208692,Low


In [29]:
claim_counts = np.zeros(n_members)

In [30]:
claim_counts[members ["risk_level"]=="Low"] = np.random.randint(6,9, size = (members["risk_level"] =="Low").sum())

In [31]:
claim_counts[members ["risk_level"]=="Medium"] = np.random.randint (12,16, size = (members["risk_level"] =="Medium").sum())

In [32]:
claim_counts[members ["risk_level"]=="High"] = np.random.randint (20,31, size = (members["risk_level"] =="High").sum())

In [33]:
members["annual_claims"] = claim_counts.astype(int)

In [34]:
claims =  members.loc [members.index.repeat (members["annual_claims"])].copy()

In [35]:
print("Check annual claims mean BEFORE expansion:")
print(members["annual_claims"].mean())

Check annual claims mean BEFORE expansion:
10.42802


In [36]:
claim_counts = np.zeros(n_members)

low_mask = members["risk_level"] == "Low"
med_mask = members["risk_level"] == "Medium"
high_mask = members["risk_level"] == "High"

claim_counts[low_mask] = np.random.randint(6, 9, low_mask.sum())
claim_counts[med_mask] = np.random.randint(12, 16, med_mask.sum())
claim_counts[high_mask] = np.random.randint(20, 31, high_mask.sum())

members["annual_claims"] = claim_counts.astype(int)

print("New avg claims per member:", members["annual_claims"].mean())
print(members["risk_level"].value_counts(normalize=True))


New avg claims per member: 10.42628
risk_level
Low       0.65
Medium    0.25
High      0.10
Name: proportion, dtype: float64


In [37]:
claims = members.loc[
    members.index.repeat(members["annual_claims"])
].copy()

print("Total claims rows:", len(claims))


Total claims rows: 521314


In [38]:
claims["claim_id"]= ["C" + str(i) for i in range (len(claims))]

In [39]:
claims .sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id
34027,M34027,57,M,South,3,0,6.431105,High,27,C353874
44261,M44261,60,F,North,3,0,6.571159,High,24,C462184
1566,M1566,52,F,Central,2,0,4.294092,High,30,C16557
19311,M19311,56,M,East,2,0,5.029735,High,22,C201874
39108,M39108,63,F,Central,3,0,4.820653,High,30,C408143
36404,M36404,33,M,East,0,0,0.134822,Low,8,C378935
18426,M18426,16,F,Costal,0,0,-1.306694,Low,6,C192615
22222,M22222,62,M,North,2,0,5.049247,High,26,C232077
35644,M35644,61,F,East,2,0,3.306052,Medium,14,C370807
19318,M19318,29,M,North,0,1,1.445071,Low,8,C201962


In [40]:
len(claims)

521314

In [41]:
members["annual_claims"].sum()

np.int64(521314)

In [42]:
start_date = datetime(2024,1,1)

In [43]:
claims["date"] = [
    start_date + timedelta(days=np.random.randint(0, 365))
    for _ in range(len(claims))
]


In [44]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date
30069,M30069,60,F,Central,3,0,8.076006,High,21,C313198,2024-09-20
41837,M41837,31,M,West,0,1,0.673464,Low,8,C436622,2024-03-06
41229,M41229,1,F,West,0,0,1.434113,Low,6,C430318,2024-02-11
1053,M1053,40,F,West,2,1,4.348566,High,25,C10985,2024-01-26
5425,M5425,59,M,North,2,0,3.956528,High,29,C57013,2024-07-20
11546,M11546,27,M,Costal,0,0,0.197715,Low,7,C120421,2024-03-25
27611,M27611,50,F,East,2,0,4.608335,High,22,C287989,2024-06-08
25983,M25983,32,M,Costal,0,0,1.314218,Low,8,C271571,2024-12-14
23926,M23926,61,F,South,2,0,4.773487,High,21,C250296,2024-09-19
16652,M16652,75,F,South,0,0,1.238112,Low,7,C173933,2024-11-07


In [45]:
claims["date"].min()

Timestamp('2024-01-01 00:00:00')

In [46]:
claims["date"].max()

Timestamp('2024-12-30 00:00:00')

In [47]:
base_pos = np.random.choice (["OP","ER","IP"], size= len(claims), p =[0.65,0.22, 0.13])

In [48]:
behavioral_mask = claims["behavioral_flag"] ==1

In [49]:
er_boost = np.random.rand (len (claims))<0.20

In [50]:
base_pos[(behavioral_mask)& (er_boost)] ="ER"

In [51]:
claims["place_of_service"] = base_pos

In [52]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date,place_of_service
37718,M37718,25,M,West,1,0,0.865713,Low,6,C393358,2024-05-22,ER
24220,M24220,60,M,Costal,1,1,3.767254,Medium,15,C253312,2024-03-22,OP
13722,M13722,36,M,East,1,0,2.796518,Medium,14,C143097,2024-07-20,OP
40018,M40018,41,F,South,0,0,-0.630609,Low,7,C417693,2024-10-29,OP
25828,M25828,66,F,Costal,1,0,3.651791,Medium,15,C270118,2024-03-09,OP
45620,M45620,39,F,West,1,0,2.399645,Medium,12,C476130,2024-12-06,IP
32497,M32497,55,M,Central,2,0,4.961961,High,22,C338079,2024-09-19,IP
44681,M44681,26,M,Central,0,0,-0.442306,Low,7,C466496,2024-08-01,IP
14043,M14043,42,F,South,2,1,4.813782,High,22,C146620,2024-01-22,OP
47347,M47347,44,M,South,1,0,0.569503,Low,6,C493759,2024-10-04,OP


In [53]:
claims["place_of_service"].value_counts(normalize= True)

,proportion
place_of_service,
OP,0.627716
ER,0.247179
IP,0.125105


In [54]:
pd.crosstab(claims["behavioral_flag"],claims["place_of_service"], normalize="index")

place_of_service,ER,IP,OP
behavioral_flag,,,
0,0.220175,0.129725,0.650100
1,0.375899,0.103084,0.521017


In [55]:
claims["diagnosis_group"] = np.random.choice(["low_acuity", "chronic","severe"], size =len(claims), p=[0.4,0.4,0.2])

In [56]:
claims ["admission_flag"] = 0
claims.loc [(claims ["place_of_service"] =="IP"), "admission_flag"] = 1

In [57]:
claims.loc [(claims ["place_of_service"] =="ER") & (claims["diagnosis_group"]=="severe"), "admission_flag"] = 1

In [58]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date,place_of_service,diagnosis_group,admission_flag
908,M908,46,F,East,1,0,1.402461,Low,8,C9423,2024-01-21,IP,low_acuity,1
36758,M36758,55,M,Central,0,0,1.133196,Low,7,C382777,2024-01-08,OP,severe,0
24917,M24917,26,M,South,0,0,-0.361819,Low,7,C260713,2024-03-12,OP,chronic,0
49551,M49551,54,F,Central,4,1,10.222529,High,24,C516720,2024-08-30,IP,chronic,1
28570,M28570,72,M,Costal,3,0,5.559900,High,28,C297893,2024-11-10,IP,chronic,1
2514,M2514,62,F,West,0,0,-2.093396,Low,8,C26692,2024-04-13,ER,severe,1
3673,M3673,68,M,West,1,0,1.575848,Low,8,C38595,2024-11-24,OP,chronic,0
38912,M38912,58,F,Central,0,0,-0.985617,Low,8,C406047,2024-01-28,ER,severe,1
20113,M20113,60,F,West,2,1,6.558848,High,30,C210242,2024-09-23,ER,severe,1
23626,M23626,51,F,East,2,1,3.339223,Medium,12,C247211,2024-03-23,OP,low_acuity,0


In [59]:
claims["admission_flag"].mean()

np.float64(0.17446682805372576)

In [60]:
pd.crosstab(claims["place_of_service"], claims["admission_flag"])

admission_flag,0,1
place_of_service,,
ER,103125,25733
IP,0,65219
OP,327237,0


In [61]:
cost = np.zeros (len(claims))

In [62]:
cost[claims["place_of_service"] == "OP"]= np.random.uniform(70, 300, size = (claims["place_of_service"]=="OP").sum())

In [63]:
cost[claims["place_of_service"] == "ER"]= np.random.uniform(900, 2000, size = (claims["place_of_service"]=="ER").sum())

In [64]:
cost[claims["place_of_service"] == "IP"]= np.random.uniform(8000, 35000, size = (claims["place_of_service"]=="IP").sum())

In [65]:
high_mask = claims ["risk_level"] =="High"

In [66]:
cost[high_mask]  *= np.random.uniform (2.5, 5.0, size = high_mask.sum())

In [67]:
claims["allowed_amount"] = cost

In [68]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date,place_of_service,diagnosis_group,admission_flag,allowed_amount
38104,M38104,25,F,Costal,0,0,0.767342,Low,7,C397410,2024-06-22,OP,low_acuity,0,249.384275
27230,M27230,7,M,West,0,0,-1.295989,Low,7,C284237,2024-11-17,OP,low_acuity,0,178.321828
46727,M46727,56,M,North,0,0,0.447771,Low,7,C487689,2024-01-31,OP,severe,0,260.071097
5439,M5439,33,F,South,2,0,3.912245,High,29,C57209,2024-11-16,ER,chronic,0,3979.360628
36094,M36094,52,M,East,1,0,1.966261,Medium,12,C375757,2024-11-18,OP,chronic,0,286.690876
49461,M49461,73,M,South,1,0,2.474731,Medium,14,C515749,2024-07-24,OP,chronic,0,109.548917
49233,M49233,34,F,Central,0,1,0.638106,Low,8,C513292,2024-04-20,OP,chronic,0,159.249666
14116,M14116,43,M,Central,3,0,7.125841,High,27,C147337,2024-02-15,IP,low_acuity,1,53351.161554
42436,M42436,33,F,West,1,0,2.180215,Medium,14,C442897,2024-07-17,OP,severe,0,222.019576
15362,M15362,26,M,West,0,0,-0.244862,Low,6,C160633,2024-11-24,OP,severe,0,143.061656


In [69]:
claims.groupby ("place_of_service")["allowed_amount"].mean()

,allowed_amount
place_of_service,
ER,2447.209850
IP,35484.526734
OP,305.358327


In [70]:
claims.groupby("risk_level")["allowed_amount"].mean()



/tmp/ipython-input-1427283996.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  claims.groupby("risk_level")["allowed_amount"].mean()


,allowed_amount
risk_level,
Low,3190.822461
Medium,3157.612894
High,11748.036635


In [71]:

shock_members =  np.random.choice (members["member_id"], size = int (0.015 * n_members), replace = False)

In [72]:
shock_mask = claims ["member_id"].isin (shock_members) & \
 (claims["place_of_service"] == "IP")

In [73]:
claims.loc [shock_mask, "allowed_amount"]*= np.random.uniform (4,8)

In [74]:
claims["allowed_amount"].describe()

,allowed_amount
count,5.213140e+05
mean,5.540437e+03
std,1.941075e+04
min,7.000014e+01
25%,1.899124e+02
50%,3.697050e+02
75%,1.625971e+03
max,1.020391e+06


In [75]:
claims.groupby("member_id")["allowed_amount"].sum().describe()

,allowed_amount
count,5.000000e+04
mean,5.776614e+04
std,1.145118e+05
min,5.860675e+02
25%,6.249101e+03
50%,2.817106e+04
75%,5.187808e+04
max,3.346453e+06


In [76]:
claims["avoidable_er"] = np.where(
    (claims["place_of_service"] =="ER") &
    (claims["admission_flag"] ==0) &
    (claims["diagnosis_group"] == "low_acuity"),
    1,0
)

In [77]:
claims ["avoidable_er"].mean()

np.float64(0.09870442765780317)

In [78]:
claims.groupby ("place_of_service")["avoidable_er"].mean()

,avoidable_er
place_of_service,
ER,0.399323
IP,0.000000
OP,0.000000


In [79]:
n_provider = 1000

In [80]:
provider_ids = np.array([f"P{i}" for i in range (n_provider)])

In [81]:
specialities = np.random.choice (["PrimaryCare", "Emergency","Specialist" ], size= n_provider, p =[0.35,0.15, 0.50])

In [82]:
provider_regions = np.random.choice (["North", "South", "East", "West", "Central", "Costal"], size = n_provider)

In [83]:
providers = pd.DataFrame({"provider_id":provider_ids ,"speciality":specialities , "provider_region":provider_regions})

In [84]:
providers.sample(10)


,provider_id,speciality,provider_region
844,P844,Emergency,West
969,P969,PrimaryCare,East
889,P889,Specialist,West
247,P247,Specialist,Central
24,P24,Specialist,West
353,P353,Specialist,West
64,P64,Specialist,East
466,P466,PrimaryCare,East
642,P642,PrimaryCare,West
817,P817,PrimaryCare,North


In [85]:
providers["speciality"].value_counts(normalize= True)

,proportion
speciality,
Specialist,0.499
PrimaryCare,0.350
Emergency,0.151


In [86]:
pcp_ids = providers.loc [providers ["speciality"] =="PrimaryCare", "provider_id"].values

In [87]:
er_ids = providers.loc [providers ["speciality"] =="Emergency", "provider_id"].values

In [88]:
spec_ids = providers.loc [providers ["speciality"] =="Specialist", "provider_id"].values

In [89]:
provider_assignment = [None] * len(claims)

In [90]:
provider_assignment = []
for pos in claims["place_of_service"]:
    if pos == "ER":
        provider_assignment.append(np.random.choice(er_ids))
    elif pos == "OP":
        provider_assignment.append(
            np.random.choice(np.concatenate([pcp_ids, spec_ids]))
        )
    else:
        provider_assignment.append(np.random.choice(spec_ids))

claims["provider_id"] = provider_assignment


In [91]:
print(len(provider_assignment))
print(len(claims))


521314
521314


In [92]:
claims["provider_id"] = provider_assignment

In [93]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date,place_of_service,diagnosis_group,admission_flag,allowed_amount,avoidable_er,provider_id
24713,M24713,14,F,North,0,0,1.360124,Low,8,C258523,2024-12-24,IP,chronic,1,13324.389854,0,P793
37820,M37820,55,M,South,2,0,3.639325,Medium,15,C394465,2024-07-27,ER,chronic,0,1193.977006,0,P997
49810,M49810,49,M,North,2,0,3.153644,Medium,15,C519358,2024-09-17,IP,chronic,1,30363.343378,0,P481
2119,M2119,59,M,Central,0,1,1.876527,Medium,15,C22532,2024-07-02,IP,low_acuity,1,9533.317820,0,P898
21883,M21883,44,F,South,3,1,6.970441,High,30,C228648,2024-09-19,ER,chronic,0,5042.159645,0,P696
40319,M40319,30,M,South,0,0,1.613757,Medium,15,C420961,2024-09-08,OP,low_acuity,0,91.511710,0,P51
19131,M19131,10,M,Central,1,0,3.726468,Medium,12,C199887,2024-07-11,IP,chronic,1,30909.705026,0,P906
43440,M43440,50,F,Costal,1,0,1.136445,Low,8,C453686,2024-03-30,OP,severe,0,216.132274,0,P314
23625,M23625,26,M,South,0,0,2.265286,Medium,13,C247206,2024-11-15,ER,severe,1,1921.491317,0,P355
13485,M13485,0,F,East,0,0,-1.666992,Low,6,C140551,2024-08-14,OP,chronic,0,192.257097,0,P467


In [94]:
claims = claims.merge(providers, on ="provider_id", how ="left", suffixes =("", "_provider"))

In [95]:
claims[["provider_id", "speciality"]].head()


,provider_id,speciality
0,P745,PrimaryCare
1,P339,Specialist
2,P754,PrimaryCare
3,P820,PrimaryCare
4,P914,Specialist


In [96]:
claims["speciality"].isna().sum()


np.int64(0)

In [97]:
pcp_probs = np.where(
    members["chronic_count"]>=1,
    0.70,0.45
)

In [98]:
members["pcp_engaged"] = np.random.binomial(1, pcp_probs)

In [99]:
members.groupby(
    members["chronic_count"] >= 1
)["pcp_engaged"].mean()



,pcp_engaged
chronic_count,
False,0.452221
True,0.700484


In [100]:
claims = claims.merge(
    members[["member_id", "pcp_engaged"]],
    on="member_id",
    how="left"
)

In [101]:
claims.head()


,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,date,place_of_service,diagnosis_group,admission_flag,allowed_amount,avoidable_er,provider_id,speciality,provider_region,pcp_engaged
0,M0,63,F,Central,1,0,2.835447,Medium,13,C0,2024-03-11,OP,chronic,0,75.269127,0,P745,PrimaryCare,East,1
1,M0,63,F,Central,1,0,2.835447,Medium,13,C1,2024-06-15,OP,chronic,0,210.382930,0,P339,Specialist,Central,1
2,M0,63,F,Central,1,0,2.835447,Medium,13,C2,2024-07-31,OP,low_acuity,0,242.490500,0,P754,PrimaryCare,West,1
3,M0,63,F,Central,1,0,2.835447,Medium,13,C3,2024-09-28,OP,low_acuity,0,255.505371,0,P820,PrimaryCare,East,1
4,M0,63,F,Central,1,0,2.835447,Medium,13,C4,2024-03-20,OP,chronic,0,71.131225,0,P914,Specialist,East,1


In [102]:
er_mask = claims ["place_of_service"] =="ER"

In [103]:
pcp_mask = claims  ["pcp_engaged"] == 1

In [104]:
reduced_mask =  er_mask & pcp_mask & (np.random.rand (len (claims)) <0.25)

In [105]:
claims.loc [reduced_mask, "place_of_service"]= "OP"

In [106]:
claims.loc[reduced_mask, "admission_flag"]=0

In [107]:
claims.loc [reduced_mask, "allowed_amount"] = np.random.uniform (70,300, reduced_mask.sum())


In [108]:
pd.crosstab (claims ["pcp_engaged"], claims ["place_of_service"], normalize = "index")

place_of_service,ER,IP,OP
pcp_engaged,,,
0,0.247990,0.125233,0.626778
1,0.184074,0.125016,0.690910


In [109]:
region_multipliers = {"North":1.10 , "South" :1.00 , "East":0.92 , "West": 1.05 ,"Central":0.97 ,"Costal":1.08}

In [110]:
claims ["region_multiplier"] = claims ["region"].map (region_multipliers)

In [111]:

claims ["allowed_amount"] =  claims ["allowed_amount"]* claims["region_multiplier"]

In [112]:
claims.loc[reduced_mask, "allowed_amount"] *= claims.loc[
    reduced_mask, "region_multiplier"
]

In [113]:
claims.groupby ("region")["allowed_amount"].mean().sort_values()

,allowed_amount
region,
East,5182.248113
Central,5266.105598
South,5485.827918
West,5683.216751
Costal,5767.706925
North,5938.574872


In [114]:
claims["avoidable_er"] = np.where (
    (claims ["place_of_service"] =="ER") &
    (claims["admission_flag"] ==0 ) &
    (claims ["diagnosis_group"] == "low_acuity"),
    1,0
)

In [115]:
claims.loc[claims ["place_of_service"]!= "ER", "avoidable_er"].sum()

np.int64(0)

In [116]:
print("VALIDATION SUMMARY")

VALIDATION SUMMARY


In [117]:
er_rate = (claims["place_of_service"] =="ER").mean()

In [118]:
print(er_rate)

0.21037416988609553


In [119]:
avoidable_rate = claims.loc [claims ["place_of_service"] == "ER", "avoidable_er"].mean()

In [120]:
print(avoidable_rate)

0.39970457094400524


In [121]:
member_cost = claims.groupby ("member_id")["allowed_amount"].sum()

In [122]:
top10 = member_cost.sort_values(ascending = False).head(int(0.1* len(member_cost))).sum()

In [123]:
pareto_share = top10/member_cost.sum()

In [124]:
print("top 10% of the total cost share:", round(pareto_share, 3))

top 10% of the total cost share: 0.561


In [125]:
high_risk_members = members.loc[members["risk_level"] =="High", "member_id"]

In [126]:
high_cost_share = member_cost.loc [high_risk_members].sum()/member_cost.sum()

In [127]:
print ("high risk cost  share:", round(high_cost_share,3))

high risk cost  share: 0.537


In [128]:
claims.sample(10)

,member_id,age,gender,region,chronic_count,behavioral_flag,risk_score,risk_level,annual_claims,claim_id,...,place_of_service,diagnosis_group,admission_flag,allowed_amount,avoidable_er,provider_id,speciality,provider_region,pcp_engaged,region_multiplier
109168,M10472,9,M,East,1,0,1.358253,Low,6,C109168,...,OP,severe,0,246.668169,0,P990,PrimaryCare,Central,1,0.92
74325,M7087,42,M,East,2,0,2.835702,Medium,12,C74325,...,OP,chronic,0,74.887624,0,P390,Specialist,East,1,0.92
68104,M6485,39,F,South,1,0,1.865842,Medium,13,C68104,...,OP,low_acuity,0,71.203475,0,P944,Specialist,East,1,1.00
94362,M9034,51,M,Costal,0,0,-1.176966,Low,8,C94362,...,OP,severe,0,230.557670,0,P289,PrimaryCare,Costal,1,1.08
503026,M48253,30,F,Costal,0,0,1.460312,Low,8,C503026,...,OP,severe,0,294.175343,0,P264,Specialist,South,0,1.08
325534,M31259,26,M,East,0,0,-0.283064,Low,7,C325534,...,OP,low_acuity,0,177.888047,0,P247,Specialist,Central,1,0.92
9477,M914,54,F,West,1,1,2.199134,Medium,12,C9477,...,ER,chronic,0,1630.323943,0,P382,Emergency,East,1,1.05
460586,M44112,58,M,Central,1,0,3.811083,High,23,C460586,...,ER,chronic,0,3929.078488,0,P364,Emergency,North,1,0.97
426712,M40878,18,M,East,0,0,1.062965,Low,7,C426712,...,IP,chronic,1,29207.241510,0,P913,Specialist,East,1,0.92
498866,M47846,74,F,West,3,1,7.042252,High,30,C498866,...,ER,severe,1,3784.003946,0,P743,Emergency,North,0,1.05


In [129]:
print("Risk distribution:")
print(members["risk_level"].value_counts(normalize=True))

print("\nIP share by risk:")
print(
    claims[claims["place_of_service"]=="IP"]
    .groupby("risk_level")["allowed_amount"]
    .sum()
    /
    claims["allowed_amount"].sum()
)


Risk distribution:
risk_level
Low       0.65
Medium    0.25
High      0.10
Name: proportion, dtype: float64

IP share by risk:
risk_level
Low       0.233305
Medium    0.170298
High      0.466653
Name: allowed_amount, dtype: float64


/tmp/ipython-input-2441060131.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("risk_level")["allowed_amount"]


In [130]:
claims = claims.drop(columns=["risk_score"], errors="ignore")

members = members.drop(columns=["risk_score"], errors="ignore")


In [131]:
claims.to_parquet("claims_phase1.parquet", index=False)
members.to_parquet("members_phase1.parquet", index=False)
providers.to_parquet("providers_phase1.parquet", index=False)


In [132]:
from google.colab import files

files.download("claims_phase1.parquet")
files.download("members_phase1.parquet")
files.download("providers_phase1.parquet")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [133]:
from google.colab import drive
drive.mount('/content/drive')

claims.to_parquet("/content/drive/MyDrive/claims_phase1.parquet", index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [134]:
import json

summary = {
    "n_members": len(members),
    "n_claims": len(claims),
    "avg_claims_per_member": members["annual_claims"].mean(),
    "risk_distribution": members["risk_level"].value_counts(normalize=True).to_dict(),
    "top10_cost_share": float(
        claims.groupby("member_id")["allowed_amount"].sum()
        .sort_values(ascending=False)
        .head(int(0.1 * len(members)))
        .sum()
        /
        claims["allowed_amount"].sum()
    )
}

with open("phase1_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

files.download("phase1_summary.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>